In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os

from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:
# Task 1: Write your code here:
food_path = os.path.join(path, 'Q1_data.csv')
df_food = pd.read_csv(food_path)

print(f"Dataset shape: {df_food.shape}")

In [ ]:
# Task 2: Write your code here:
df_food.head()

In [ ]:
# Task 3: Write your code here:
df_food.info()

In [ ]:
# Task 4: Write your code here:
df_food.describe()

In [ ]:
# Task 5: Write your code here:
plt.figure(figsize=(10, 5))
plt.hist(df_food['Delivery_Time'], bins=30, edgecolor='black')
plt.title('Delivery Time Distribution')
plt.ylabel('Delivery Time')
plt.show()

In [ ]:
# Task 1: Write your code here:
df_food = df_food.drop(columns="Order_ID", axis=1)
df_food.head()

In [ ]:
# Task 2: Write your code here:
print("Missing values:")
print(df_food.isnull().sum())
# to hande mising values first for categorical variables ['Traffic_Level', 'Weather', 'Time_of_Day'] i choose to see value counts for each variable and impote it with the most value repated
print()
print("Weather variable: ")
print(df_food['Weather'].value_counts())
df_food['Weather'] = df_food['Weather'].fillna('Clear')

print()
print("Traffic_Level variable: ")
print(df_food['Traffic_Level'].value_counts())
df_food['Traffic_Level'] = df_food['Traffic_Level'].fillna('Medium')

print()
print("Time_of_Day variable: ")
print(df_food['Time_of_Day'].value_counts())
df_food['Time_of_Day'] = df_food['Time_of_Day'].fillna('Morning')

print()
#for numaric variables: [Delivery_Time, Courier_Experience_yrs] i cboose to impot it with the mean
df_food['Delivery_Time'] = df_food['Delivery_Time'].fillna(df_food['Delivery_Time'].mean())
df_food['Courier_Experience_yrs'] = df_food['Courier_Experience_yrs'].fillna(df_food['Courier_Experience_yrs'].mean())

print("Missing values:")
print(df_food.isnull().sum())

In [ ]:
# Task 3: Write your code here:
duplicates = df_food.duplicated().sum()
print(f"Number of Duplicate Samples: {duplicates}")
print(duplicates)
if duplicates > 0:
  print("Dropping Duplicates...")
  df_food.drop_duplicates(inplace=True)
  print("Duplicates Dropped.")
else:
  print("No Duplicate Samples Found.")

duplicates = df_food.duplicated().sum()
print(f"Number of Duplicate Samples: {duplicates}")

In [ ]:
# Task 4: Write your code here:
df_food_copy = df_food.copy()
#categories = df_food_copy[['Weather', 'Traffic_Level', 'Time_of_Day', 'Vehicle_Type']]
encoder = LabelEncoder()

#df_food_copy[categories] = encoder.fit_transform(df_food_copy[categories])
df_food_copy['Weather'] = encoder.fit_transform(df_food_copy['Weather'])
df_food_copy['Traffic_Level'] = encoder.fit_transform(df_food_copy['Traffic_Level'])
df_food_copy['Time_of_Day'] = encoder.fit_transform(df_food_copy['Time_of_Day'])
df_food_copy['Vehicle_Type'] = encoder.fit_transform(df_food_copy['Vehicle_Type'])

df_food_copy.head()

In [ ]:
# Task 5: Write your code here:
standard_scaler = StandardScaler() # Instantiate StandardScaler
data_for_scale = df_food_copy.copy()
features = data_for_scale.columns
data_for_scale[features] = standard_scaler.fit_transform(data_for_scale[features]) # Apply fit_transform

data_for_scale.head()

In [ ]:
# Task 6: Write your code here:
import seaborn as sns
def check_target_imbalance(df, target_column):
    print("Target Distribution:")
    print(df[target_column].value_counts())
    sns.countplot(x=df[target_column])
    plt.title("Target Distribution")
    plt.show()

check_target_imbalance(data_for_scale, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
X = data_for_scale.drop("Delivery_Time",axis=1)
y = data_for_scale['Delivery_Time']

In [ ]:
# Task 2,3,4,5: Write your code here:
n_splits = 5
model = RandomForestRegressor()
kf = KFold(n_splits=5, shuffle=True, random_state=42)
y_pred=0

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # Train
  model.fit(X_train, y_train)

  # Predict
  y_pred = model.predict(X_test)

  # Calculate metrics
  mae = mean_absolute_error(y_test, y_pred)
print(mae)

In [ ]:
# Task 1: Write your code here:
model_importance = list(zip(X.columns, model.feature_importances_))
sorted_catboost_importance = sorted(model_importance, key=lambda x: x[1], reverse=True)

# Extract features and their importances
features, importances = zip(*sorted_catboost_importance)

# Plot feature importances
plt.figure(figsize=(18, 14))
plt.barh(features, importances, color='orange')
plt.xlabel('Importance Score')
plt.ylabel('Features')
plt.title('Random forest Feature Importance')
plt.gca().invert_yaxis()  # Invert y-axis to show the most important features at the top
plt.show()

In [ ]:
# Task 2: Write your code here:
plt.figure(figsize=(10, 6))
plt.hist(y_pred, bins=30, edgecolor='black')
plt.title('Distribution of Predictions')
plt.xlabel('Predicted Emission')
plt.ylabel('Count')
plt.show()

In [ ]:
# Task Bonus: Write your code here: